# Qualitative Human–Model Comparison

This notebook displays a compact set of qualitative comparison tables using the refreshed human baseline rankings and the current blind-instruction model responses.

Focus:
- stable shared human priors
- anchor-sensitive human priors
- low-consensus pluralistic human questions
- how model groups differ relative to the human response distribution

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path('..').resolve().parent
HUMAN_DIR = ROOT / 'analysis' / 'human_answers'
EXPORTS_DIR = ROOT / 'analysis' / 'session2' / 'exports'

hh_C = pd.read_csv(HUMAN_DIR / 'hh_ranked_vC.csv')
hh_B = pd.read_csv(HUMAN_DIR / 'hh_ranked_vB.csv')
hh_A = pd.read_csv(HUMAN_DIR / 'hh_ranked_vA.csv')
model_df = pd.read_csv(EXPORTS_DIR / 'responses_model_inst_blind.csv')

GROUP_LABELS = {
    'VLM': 'VLM',
    'VLM backbone decoder': 'Backbone decoder',
    'standalone LLM': 'Standalone LLM',
    'standalone LLM (think)': 'Standalone LLM (think)',
}

VARIANT_LABELS = {'C': 'Original', 'B': 'Weaker-object', 'A': 'Pronominalized'}

HUMAN_ID_COLS = [c for c in hh_C.columns if c not in ['question_id', 'rank', 'question_en', 'op', 'ent', 'hh_sbert', 'entropy', 'accuracy', 'gt']]

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

print('Loaded:')
print('  human rows C/B/A:', len(hh_C), len(hh_B), len(hh_A))
print('  model rows:', len(model_df))
print('  model groups:', sorted(model_df['model_group'].dropna().unique().tolist()))

Loaded:
  human rows C/B/A: 113 113 113
  model rows: 12506
  model groups: ['VLM', 'VLM backbone decoder', 'standalone LLM', 'standalone LLM (think)']


In [2]:
CASE_STUDIES = [
    'How many bottles of water are on the table?',
    'Do the horses love each other?',
    'What type of bear is this?',
    'What kind of information does the yellow and white book look like it contains?'
]

MODEL_ORDER = [
    'InternVL-8B', 'InternVL-8B (LM)',
    'LLaVA-1.5-7B', 'LLaVA-1.5 (LM)',
    'Qwen3-VL-8B', 'Qwen3-VL-8B (LM)',
    'Qwen2.5-7B-Instruct', 'Qwen3-8B', 'Qwen3-8B (think)', 'Mistral-7B'
]

def top_human_answers(row, n=10):
    vals = [str(row[c]).strip() for c in HUMAN_ID_COLS if pd.notna(row[c]) and str(row[c]).strip()]
    vc = pd.Series(vals).value_counts()
    return '; '.join([f'{k} ({v})' for k, v in vc.head(n).items()])

QUESTION_ID_BY_TEXT = hh_C.set_index('question_en')['question_id'].to_dict()

def get_human_row(df, q, variant='C'):
    if variant == 'C':
        sub = df[df['question_en'] == q]
        return sub.iloc[0] if len(sub) else None
    qid = QUESTION_ID_BY_TEXT.get(q)
    if qid is None:
        return None
    sub = df[df['question_id'] == qid]
    return sub.iloc[0] if len(sub) else None

def baseline_table(questions):
    rows = []
    for q in questions:
        rC = get_human_row(hh_C, q, variant='C')
        if rC is None:
            continue
        rA = get_human_row(hh_A, q, variant='A')
        rows.append({
            'Question': q,
            'Op': rC['op'],
            'Entity': rC['ent'],
            'HH (Original)': round(float(rC['hh_sbert']), 3),
            'HH (Pronominalized)': round(float(rA['hh_sbert']), 3) if rA is not None else None,
            'Δ Original→Pronom.': round(float(rC['hh_sbert']) - float(rA['hh_sbert']), 3) if rA is not None else None,
            'Top Human Answers (Original)': top_human_answers(rC, n=8),
            'Top Human Answers (Pronominalized)': top_human_answers(rA, n=8) if rA is not None else None,
        })
    return pd.DataFrame(rows)

def per_model_table(question, variants=('C',)):
    sub = model_df[(model_df['question_en'] == question) & (model_df['variant'].isin(list(variants)))].copy()
    if sub.empty:
        return pd.DataFrame()
    sub['Group'] = sub['model_group'].map(GROUP_LABELS)
    sub['Variant'] = sub['variant'].map(VARIANT_LABELS)
    sub['model_rank'] = sub['model'].apply(lambda x: MODEL_ORDER.index(x) if x in MODEL_ORDER else 999)
    sub = sub.sort_values(['variant', 'model_rank', 'model'])
    out = sub[['Variant', 'Group', 'model', 'response']].rename(columns={'model': 'Model', 'response': 'Response'})
    return out.reset_index(drop=True)

def grouped_response_table(question, variant='C', topn=5):
    sub = model_df[(model_df['question_en'] == question) & (model_df['variant'] == variant)].copy()
    rows = []
    for grp, g in sub.groupby('model_group'):
        vc = g['response'].astype(str).str.strip().value_counts().head(topn)
        rows.append({
            'Variant': VARIANT_LABELS[variant],
            'Group': GROUP_LABELS.get(grp, grp),
            'Top Group Responses': '; '.join([f'{k} ({v})' for k, v in vc.items()])
        })
    return pd.DataFrame(rows)

def grouped_response_table_multi(question, variants=('C', 'A'), topn=4):
    frames = [grouped_response_table(question, variant=v, topn=topn) for v in variants]
    return pd.concat(frames, ignore_index=True)


## 1. Human Baseline Summary

In [3]:
display(baseline_table(CASE_STUDIES))

,Question,Op,Entity,HH (Original),HH (Pronominalized),Δ Original→Pronom.,Top Human Answers (Original),Top Human Answers (Pronominalized)
0,How many bottles of water are on the table?,count,object,0.783,0.662,0.121,1 (16); 3 (11); 2 (10); 44 bottles (1); 4 (1); 9 bottles (1),2 (10); 4 (5); 1 (4); 5 (3); 10 (3); 0 (3); 3 (3); 6 (2)
1,Do the horses love each other?,act,animal,0.770,0.843,-0.072,yes (27); no (8); not really (1); They do. (1); I don't know. (1); likes (1); Yes. (1),"yes (31); no (6); Yes, they do. (1); likes (1); No. (1)"
2,What type of bear is this?,ident,animal,0.674,0.341,0.333,brown bear (10); polar bear (9); asian black bear (8); Grizzly (2); arctic (2); Scary (1); Moon bear (1); wild bear (1),dog (6); food (5); animal (3); Electronic device (2); vegetable (2); furniture (1); Dessert (1); Plant type (1)
3,What kind of information does the yellow and white book look like it contains?,ident,other,0.183,0.169,0.014,cooking (2); Environment (1); recipe (1); food (1); Psychology (1); investment advice (1); food recipes (1); travell...,Color information (1); Treasure map location (1); warning sign (1); instructions for use (1); Science (1); food info...


## 2. Per-Model Blind Answers on the Original Questions

In [4]:
for q in CASE_STUDIES:
    display(Markdown(f'### {q}'))
    display(per_model_table(q, variants=('C',)))

### How many bottles of water are on the table?

,Variant,Group,Model,Response
0,Original,VLM,InternVL-8B,2
1,Original,Backbone decoder,InternVL-8B (LM),3
2,Original,VLM,LLaVA-1.5-7B,0
3,Original,Backbone decoder,LLaVA-1.5 (LM),0
4,Original,VLM,Qwen3-VL-8B,0
5,Original,Backbone decoder,Qwen3-VL-8B (LM),0
6,Original,Standalone LLM,Qwen2.5-7B-Instruct,5
7,Original,Standalone LLM,Qwen3-8B,6
8,Original,Standalone LLM (think),Qwen3-8B (think),6
9,Original,Standalone LLM,Mistral-7B,i cannot answer that question definitively without knowing exact number of bottles on table


### Do the horses love each other?

,Variant,Group,Model,Response
0,Original,VLM,InternVL-8B,yes
1,Original,Backbone decoder,InternVL-8B (LM),yes
2,Original,VLM,LLaVA-1.5-7B,yes
3,Original,Backbone decoder,LLaVA-1.5 (LM),yes
4,Original,VLM,Qwen3-VL-8B,yes
5,Original,Backbone decoder,Qwen3-VL-8B (LM),no
6,Original,Standalone LLM,Qwen2.5-7B-Instruct,yes
7,Original,Standalone LLM,Qwen3-8B,yes
8,Original,Standalone LLM (think),Qwen3-8B (think),yes
9,Original,Standalone LLM,Mistral-7B,horses form strong social bonds but they do not experience love in same way humans do they form herd relationships a...


### What type of bear is this?

,Variant,Group,Model,Response
0,Original,VLM,InternVL-8B,grizzly bear
1,Original,Backbone decoder,InternVL-8B (LM),grizzly bear
2,Original,VLM,LLaVA-1.5-7B,black
3,Original,Backbone decoder,LLaVA-1.5 (LM),brown
4,Original,VLM,Qwen3-VL-8B,black bear
5,Original,Backbone decoder,Qwen3-VL-8B (LM),grizzly bear
6,Original,Standalone LLM,Qwen2.5-7B-Instruct,grizzly bear
7,Original,Standalone LLM,Qwen3-8B,brown bear
8,Original,Standalone LLM (think),Qwen3-8B (think),brown bear
9,Original,Standalone LLM,Mistral-7B,without image or additional context it is impossible to determine what type of bear is being referred to please prov...


### What kind of information does the yellow and white book look like it contains?

,Variant,Group,Model,Response
0,Original,VLM,InternVL-8B,text
1,Original,Backbone decoder,InternVL-8B (LM),recipes
2,Original,VLM,LLaVA-1.5-7B,science
3,Original,Backbone decoder,LLaVA-1.5 (LM),dictionary
4,Original,VLM,Qwen3-VL-8B,recipes
5,Original,Backbone decoder,Qwen3-VL-8B (LM),instructions
6,Original,Standalone LLM,Qwen2.5-7B-Instruct,textbook
7,Original,Standalone LLM,Qwen3-8B,phone directory
8,Original,Standalone LLM (think),Qwen3-8B (think),phone directory
9,Original,Standalone LLM,Mistral-7B,yellow and white book is commonly known as house of commons procedure and practice so it likely contains parliamenta...


## 3. Group-Level Response Tendencies for the Same Questions

These tables are shown only as qualitative summaries of dominant response patterns, not as standalone evidence for group-level claims.

In [5]:
for q in CASE_STUDIES:
    display(Markdown(f'### {q}'))
    display(grouped_response_table_multi(q, variants=('C', 'A'), topn=4))

### How many bottles of water are on the table?

,Variant,Group,Top Group Responses
0,Original,VLM,0 (9); 2 (1); question asks for number of bottles of water on table but no image is provided since instruction says ...
1,Original,Backbone decoder,0 (7); 3 (2); 8 (1); 2 (1)
2,Original,Standalone LLM,0 (3); 5 (2); i cannot answer that question definitively without knowing exact number of bottles on table (1); 1 or ...
3,Original,Standalone LLM (think),empty (1); think okay let's see question is asking how many bottles of water are on table user wants answer in singl...


### Do the horses love each other?

,Variant,Group,Top Group Responses
0,Original,VLM,yes (10); no (1)
1,Original,Backbone decoder,yes (8); no (3)
2,Original,Standalone LLM,yes (5); horses form strong social bonds but they do not experience love in same way humans do they form herd relati...
3,Original,Standalone LLM (think),yes (2); horse (1); think okay let's see question is do horses love each other and i need to answer with single word...


### What type of bear is this?

,Variant,Group,Top Group Responses
0,Original,VLM,black (4); polar bear (3); brown (1); grizzly bear (1)
1,Original,Backbone decoder,brown bear (4); grizzly bear (2); brown (2); grizzly (2)
2,Original,Standalone LLM,polar bear (3); grizzly bear (2); without image or additional context it is impossible to determine what type of bea...
3,Original,Standalone LLM (think),brown bear (2); bear (1); think okay user is asking what type of bear is this and they want answer in single word or...


### What kind of information does the yellow and white book look like it contains?

,Variant,Group,Top Group Responses
0,Original,VLM,recipes (2); textbook (1); information about books (1); text (1)
1,Original,Backbone decoder,it contains information about yellow and white book (1); information about book (1); recipes (1); dictionary (1)
2,Original,Standalone LLM,yellow and white book is commonly known as house of commons procedure and practice so it likely contains parliamenta...
3,Original,Standalone LLM (think),book (1); think okay let's see user is asking about what kind of information yellow and white book looks like it con...


## 4. Additional Question Comparisons

In [6]:
EXTRA_QUESTIONS = [
    'How many of these people are female?',
    'What type of shoe is that?'
]

for q in EXTRA_QUESTIONS:
    display(Markdown(f'### {q}'))
    display(per_model_table(q, variants=('C',)))
    display(grouped_response_table_multi(q, variants=('C',), topn=5))

### How many of these people are female?

,Variant,Group,Model,Response
0,Original,VLM,InternVL-8B,1
1,Original,Backbone decoder,InternVL-8B (LM),since no specific image or details are provided i will assume common scenario where there are 2 people 1 male and 1 ...
2,Original,VLM,LLaVA-1.5-7B,0
3,Original,Backbone decoder,LLaVA-1.5 (LM),1
4,Original,VLM,Qwen3-VL-8B,0
5,Original,Backbone decoder,Qwen3-VL-8B (LM),0
6,Original,Standalone LLM,Qwen2.5-7B-Instruct,cannot determine
7,Original,Standalone LLM,Qwen3-8B,half
8,Original,Standalone LLM (think),Qwen3-8B (think),half
9,Original,Standalone LLM,Mistral-7B,without knowing number or identities of people in question it is impossible to provide accurate answer using single ...


,Variant,Group,Top Group Responses
0,Original,VLM,0 (9); 1 (1); question asks how many of people in imagined image are female but no specific image is provided since ...
1,Original,Backbone decoder,1 (5); 0 (4); since no specific image or details are provided i will assume common scenario where there are 2 people...
2,Original,Standalone LLM,half (2); without knowing number or identities of people in question it is impossible to provide accurate answer usi...
3,Original,Standalone LLM (think),half (2); female (1); think okay let's see question is asking how many of these people are female but wait problem s...


### What type of shoe is that?

,Variant,Group,Model,Response
0,Original,VLM,InternVL-8B,sneaker
1,Original,Backbone decoder,InternVL-8B (LM),sneaker
2,Original,VLM,LLaVA-1.5-7B,sneaker
3,Original,Backbone decoder,LLaVA-1.5 (LM),sneaker
4,Original,VLM,Qwen3-VL-8B,sneaker
5,Original,Backbone decoder,Qwen3-VL-8B (LM),sneaker
6,Original,Standalone LLM,Qwen2.5-7B-Instruct,sneaker
7,Original,Standalone LLM,Qwen3-8B,sneaker
8,Original,Standalone LLM (think),Qwen3-8B (think),sneaker
9,Original,Standalone LLM,Mistral-7B,without image or additional context it is impossible to provide accurate answer please provide more information for ...


,Variant,Group,Top Group Responses
0,Original,VLM,sneaker (9); sneakers (1); boots (1)
1,Original,Backbone decoder,sneaker (9); sneakers (1); shoe in image is sneaker (1)
2,Original,Standalone LLM,sneaker (5); without image or additional context it is impossible to provide accurate answer please provide more inf...
3,Original,Standalone LLM (think),sneaker (2); boots (1); think okay user is asking for type of shoe based on description but there's no image provide...


## 5. Quick Reading Guide

- High-consensus human questions should show concentrated human answer summaries.
- Anchor-sensitive identity questions should show large human `Original→Pronominalized` drops.
- VLMs often commit to narrow benchmark-like defaults.
- Backbone decoders often preserve similar answer spaces but more closely track human priors.
- Standalone LLMs often hedge, abstract away from the question, or over-elaborate.
- Think variants can destabilize the short-answer manifold instead of improving human alignment.